# Module 4: Search in Azure DocumentDB

**Time**: ~75 min  
**Environment**: Jupyter notebook in VS Code

This notebook is fully runnable. Enter your Azure DocumentDB connection string in Step 0, then run each cell in order. You will load sample documents, create vector and full-text indexes, run vector search, BM25 keyword search, fuzzy search, phrase search, and combine keyword + vector results with Reciprocal Rank Fusion (RRF).

> Full-text search in Azure DocumentDB is currently in gated preview. Vector DiskANN requires an M30 or higher cluster tier.


## Step 0: Connect to Azure DocumentDB

This cell installs `pymongo` if needed, asks for your connection string if `DOCUMENTDB_CONNECTION_STRING` is not set, and opens the `docdbworkshop.workshop_content` collection.

In [ ]:
import importlib.util, subprocess, sys, os, getpass
if importlib.util.find_spec("pymongo") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pymongo"])
from pymongo import MongoClient

connection_string = os.environ.get("DOCUMENTDB_CONNECTION_STRING") or getpass.getpass("Paste Azure DocumentDB connection string: ")
client = MongoClient(connection_string)
db = client["docdbworkshop"]
collection = db["workshop_content"]
print(db.command({"ping": 1}))

## Step 1: Load sample search documents

The lab uses small, readable vectors so you can focus on query syntax. Production apps would generate embeddings with your model and store the resulting vector array on each document.

In [ ]:
collection.drop()
docs = [
    {"_id":"doc-search-001","title":"DiskANN vector indexing","category":"vector","body":"Azure DocumentDB supports DiskANN vector indexes for high recall semantic similarity search over embeddings stored with documents.","sku":"SEARCH-VEC-001","embedding":[0.92,0.80,0.18]},
    {"_id":"doc-search-002","title":"BM25 keyword search","category":"full-text","body":"Azure DocumentDB full-text search ranks keyword matches with BM25 and exposes scores through searchScore metadata.","sku":"SEARCH-FTS-001","embedding":[0.20,0.12,0.94]},
    {"_id":"doc-search-003","title":"Hybrid search with RRF","category":"hybrid","body":"Hybrid search combines BM25 keyword results with vector results and fuses the ranked lists using Reciprocal Rank Fusion.","sku":"SEARCH-HYB-001","embedding":[0.76,0.70,0.42]},
    {"_id":"doc-search-004","title":"RAG grounding","category":"rag","body":"Retrieval augmented generation retrieves relevant chunks from Azure DocumentDB and grounds the model answer in that context.","sku":"RAG-PIPE-001","embedding":[0.82,0.74,0.36]},
    {"_id":"doc-search-005","title":"Operational filtering","category":"filters","body":"Search applications often filter by status, tenant, region, stock, or category after the search stage narrows candidate documents.","sku":"SEARCH-FLT-001","embedding":[0.35,0.30,0.82]}
]
collection.insert_many(docs)
print("Loaded documents:", collection.count_documents({}))

## Step 2: Create the vector index

Azure DocumentDB vector search uses `createIndexes` with key type `cosmosSearch`. This creates a DiskANN index over the `embedding` field using cosine similarity.

In [ ]:
db.command({
    "createIndexes": "workshop_content",
    "indexes": [{
        "name": "idx_embedding_diskann",
        "key": {"embedding": "cosmosSearch"},
        "cosmosSearchOptions": {
            "kind": "vector-diskann",
            "dimensions": 3,
            "similarity": "COS",
            "maxDegree": 32,
            "lBuild": 64
        }
    }]
})

## Step 3: Run vector search

`$search.cosmosSearch` compares the query vector to stored embeddings and returns the nearest documents. This finds semantic neighbors even when terms do not exactly match.

In [ ]:
semantic_query_vector = [0.90, 0.78, 0.22]
vector_results = list(collection.aggregate([
    {"$search": {"cosmosSearch": {"path": "embedding", "vector": semantic_query_vector, "k": 3}}},
    {"$project": {"_id": 0, "title": 1, "category": 1, "body": 1, "score": {"$meta": "searchScore"}}}
]))
vector_results

## Step 4: Create the full-text search index

Full-text search uses `createSearchIndexes`, not a legacy `{ field: "text" }` index. The index is explicit (`dynamic: false`) and maps only the `body` field.

In [ ]:
db.command({
    "createSearchIndexes": "workshop_content",
    "indexes": [{
        "name": "idx_body_fts",
        "definition": {"mappings": {"dynamic": False, "fields": {"body": {"type": "string"}}}}
    }]
})

## Step 5: Inspect the search index

Search indexes build asynchronously. Use `$listSearchIndexes` to verify the index exists and to inspect status before relying on it in an application.

In [ ]:
list(collection.aggregate([{ "$listSearchIndexes": {"name": "idx_body_fts"} }]))

## Step 6: Run BM25 keyword search

The `$search.text` operator returns BM25-ranked results. Keep `$search` first, name the index explicitly, and use a downstream `$limit`.

In [ ]:
bm25_results = list(collection.aggregate([
    {"$search": {"index": "idx_body_fts", "text": {"query": "BM25 ranking", "path": "body"}}},
    {"$limit": 5},
    {"$project": {"_id": 0, "title": 1, "category": 1, "body": 1, "score": {"$meta": "searchScore"}}}
]))
bm25_results

## Step 7: Run fuzzy search

Fuzzy search handles typos by allowing a bounded edit distance. Use it for user-facing search boxes, but avoid it for short tokens or precision-critical identifiers.

In [ ]:
fuzzy_results = list(collection.aggregate([
    {"$search": {"index": "idx_body_fts", "text": {"query": "retrival augmentd genration", "path": "body", "fuzzy": {"maxEdits": 1}}}},
    {"$limit": 5},
    {"$project": {"_id": 0, "title": 1, "body": 1, "score": {"$meta": "searchScore"}}}
]))
fuzzy_results

## Step 8: Run phrase search

Phrase search matches terms in order. `slop: 0` requires adjacent terms; larger `slop` values allow intervening words and trade precision for recall.

In [ ]:
phrase_results = list(collection.aggregate([
    {"$search": {"index": "idx_body_fts", "phrase": {"query": "Reciprocal Rank Fusion", "path": "body", "slop": 0}}},
    {"$limit": 5},
    {"$project": {"_id": 0, "title": 1, "body": 1, "score": {"$meta": "searchScore"}}}
]))
phrase_results

## Step 9: Run hybrid search with RRF

Hybrid search runs BM25 and vector retrieval, then fuses the ranked lists. RRF uses rank positions instead of raw scores because BM25 and vector scores are not directly comparable.

In [ ]:
def rrf(lists, k=60, top_n=5):
    scores = {}
    docs_by_id = {}
    for results in lists:
        for rank, doc in enumerate(results):
            doc_id = str(doc["_id"])
            docs_by_id[doc_id] = doc
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
    return [{**docs_by_id[doc_id], "rrfScore": score} for doc_id, score in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]]

user_query = "semantic retrieval for rag"
query_vector = [0.84, 0.76, 0.32]
keyword_hits = list(collection.aggregate([
    {"$search": {"index": "idx_body_fts", "text": {"query": user_query, "path": "body"}}},
    {"$limit": 5},
    {"$project": {"_id": 1, "title": 1, "source": "keyword", "score": {"$meta": "searchScore"}}}
]))
vector_hits = list(collection.aggregate([
    {"$search": {"cosmosSearch": {"path": "embedding", "vector": query_vector, "k": 5}}},
    {"$project": {"_id": 1, "title": 1, "source": "vector", "score": {"$meta": "searchScore"}}}
]))
rrf([keyword_hits, vector_hits])